# 10.06 · Production monitoring: online evals, feedback, review queues

**Module 10 · Testing, Evaluation & Observability** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2 h |
| 📋 Prerequisites | Lessons 10.01–10.05. |
| 🔑 Keys | none (optional: `LANGSMITH_API_KEY` for the 🌐 cell) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-15 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Collect** feedback from users, judges and reviewers in one shape, and know what each source is biased toward.
2. **Size** an online judge: how many traces to sample for a useful interval, and what it costs per day.
3. **Alert** on a failure rate only when it's worse than baseline beyond noise.
4. **Route** traces into a review queue (thumbs-down, judge flags, and a random slice), label them, and feed them back into the taxonomy and the dataset.
5. **Configure** the LangSmith equivalents (online evaluators with a sampling rate, feedback, annotation queues) and their retention costs.

## 📚 Official reading (verified 2026-10-15)

| Page | Read it for |
|---|---|
| [Online LLM-as-judge evaluators](https://docs.langchain.com/langsmith/online-evaluations) | Filters, **sampling rate**, backfill, weekly spend limit, decision-model judges |
| [Multi-turn online evaluators](https://docs.langchain.com/langsmith/online-evaluations-multi-turn) | Thread-level evaluation once a thread goes idle |
| [Attach user feedback](https://docs.langchain.com/langsmith/attach-user-feedback) | `create_feedback(run_id, key, score, ...)` |
| [Annotation queues](https://docs.langchain.com/langsmith/annotation-queues) | Queues, rubrics, reviewers, sending items to datasets |
| [Dashboards](https://docs.langchain.com/langsmith/dashboards) · [Alerts](https://docs.langchain.com/langsmith/alerts) | Monitoring charts and threshold alerts |
| [Usage and billing → retention auto-upgrades](https://docs.langchain.com/langsmith/usage-and-billing) | Which actions move a trace to extended retention |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langsmith>=0.14.7,<1" "openevals>=0.2.0,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langsmith", "openevals",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/10-testing-evaluation-observability/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm10-monitoring'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m10-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Offline: the Orbita Cloud desk from Project 09 (handoffs pattern, rule-based offline model).
from langgraph.checkpoint.memory import InMemorySaver

from mastery.agents.support import SupportDesk, build_handoffs_desk, make_support_tools, offline_model
from mastery.agents.support_traces import generate_inputs, reference_label, reference_note, run_traces

OUT = REPO_ROOT / "data" / "outputs" / "m10"
OUT.mkdir(parents=True, exist_ok=True)
desk = SupportDesk.load()
app = build_handoffs_desk(offline_model, make_support_tools(desk), checkpointer=InMemorySaver())
inputs = generate_inputs(60, seed=10)
traces = run_traces(app, inputs, desk)
print(len(traces), "traces;", {k: sum(t["kind"] == k for t in traces) for k in ("realistic", "edge", "adversarial")})

---
## Part 1 · Offline vs online evaluation

| | Offline (10.03–10.05) | Online (this lesson) |
|---|---|---|
| Data | A frozen dataset you chose | Whatever customers send today |
| Reference answers | Yes | No: only judges, rules and feedback |
| When | Before a release (and in CI) | Continuously, after release |
| Answers | "Is the new version better?" | "Is production still OK? What's new?" |
| Feeds | The release gate (10.08) | The next error analysis (10.01) and the dataset (10.03) |

The loop: **production traces → feedback and sampled judges → review queue → labels → taxonomy and dataset → next release's gate.**

---
## Part 2 · A production day, with user feedback

We simulate one day of traffic: 400 messages from a seed the lessons haven't used. Customers click 👍/👎 on some answers; as in real apps, few people click, and unhappy customers click more often.

In [ ]:
# 🧪 Offline: 400 production traces + simulated thumbs feedback in create_feedback's shape.
import random
from collections import Counter

from mastery.evals.monitoring import feedback_payload

production = run_traces(app, generate_inputs(400, seed=77, start=5001), desk)
truth = {t["id"]: reference_label(t) for t in production}      # what a careful reviewer would say (hidden in real life)

rng = random.Random(7)
feedback = []
for t in production:
    failed = truth[t["id"]] != "pass"
    if rng.random() < (0.30 if failed else 0.05):           # unhappy people click more
        thumbs_up = (not failed) if rng.random() < 0.9 else failed
        feedback.append(feedback_payload(t["id"], "user_thumbs", thumbs_up, source="user"))

true_rate = sum(v != "pass" for v in truth.values()) / len(truth)
down = sum(f["score"] == 0 for f in feedback)
print(f"{len(production)} traces | true failure rate {true_rate:.1%} | {len(feedback)} feedback ({len(feedback) / len(production):.0%} of traces)")
print(f"thumbs-down share of feedback: {down / len(feedback):.0%}")
print("one payload:", feedback[0])

**You should now see** `400 traces | true failure rate 31.2% | 56 feedback (14% of traces)`, a thumbs-down share of `79%` (two and a half times the true failure rate), and one payload with `run_id`, `key`, `score` and `source_info`.

### What just happened?

User feedback is **free and biased**: it over-represents unhappy customers and covers a fraction of traffic. It's a great **pointer** (every 👎 deserves a look) and a poor **rate**. For rates you need a random sample with a judge or a reviewer.

---
## Part 3 · Sizing the online judge

An online judge runs on a **sample** of traces. How big? Enough for the interval you need, no more: every judged trace costs judge calls (and, in LangSmith, upgrades that trace's retention by default).

In [ ]:
# 🧪 Offline: sample size for a target margin, a deterministic sampler, and the daily judge bill.
from mastery.evals.monitoring import judge_budget, sample_size, should_sample

for margin in (0.10, 0.05, 0.02):
    print(f"±{margin:.0%} on a ~30% failure rate -> judge {sample_size(margin, p=0.3)} traces")

rate = 0.35
sampled = [t for t in production if should_sample(t["id"], rate)]
print(f"\nsampling {rate:.0%}: {len(sampled)} of {len(production)} traces")
print("same ids picked again:", sampled == [t for t in production if should_sample(t["id"], rate)])
print("\nat 10,000 traces/day, 2 judges, 5% sample:", judge_budget(10_000, 0.05, judges=2))

**You should now see** 81, 323 and 2017 traces for ±10%, ±5% and ±2%; `150 of 400` traces sampled at 35% (the same ones every time, because the sampler hashes the trace id); and a daily bill of 1,000 judge calls, 1.5M tokens and about $0.15 at $0.10 per million tokens.

### What just happened?

The margin shrinks with the square root of the sample: halving it costs four times the traces. A fixed **sample size per day** (not a fixed rate) keeps the precision constant as traffic grows. Hash-based sampling means every worker agrees on which traces are in, and a re-run judges the same ones.

---
## Part 4 · The judge on the sample, corrected, and an alert

We reuse Lesson 10.04's `stopped_at_lookup` judge. Its TPR/TNR come from a **labelled validation set** (here a fresh labelled batch, standing in for last month's reviewed traces), so the sampled rate can be corrected and compared with the baseline.

In [ ]:
# 🧪 Offline: judge the sample, correct for the judge's errors, alert only beyond noise.
from mastery.agents.support_traces import REFERENCE_CATEGORIES, judge_view, offline_judge_model
from mastery.evals.judges import corrected_rate, make_binary_judge
from mastery.evals.monitoring import rate_alert

entry = REFERENCE_CATEGORIES["stopped_at_lookup"]
judge = make_binary_judge("stopped_at_lookup", entry["definition"], entry["not_this"], judge=offline_judge_model("stopped_at_lookup"))

def flag(t):
    inputs, outputs = judge_view(t)
    return bool(judge(inputs=inputs, outputs=outputs)["score"])

validation = run_traces(app, generate_inputs(200, seed=10, start=301), desk)       # reviewed last month
val_flags = [flag(t) for t in validation]
val_truth = [reference_label(t) == "stopped_at_lookup" for t in validation]

flags = [flag(t) for t in sampled]
print(f"judge flags {sum(flags)}/{len(flags)} sampled traces = {sum(flags) / len(flags):.1%}")
print("corrected:", corrected_rate(flags, val_flags, val_truth))
print(f"true rate today: {sum(truth[t['id']] == 'stopped_at_lookup' for t in production) / len(production):.1%}")

BASELINE_RATE = 0.10    # the same judge flagged 10% of last release's sample: compare like with like
alert, interval = rate_alert(flags, BASELINE_RATE)
print(f"\nalert vs baseline {BASELINE_RATE:.0%}: {alert} (judge-rate interval {interval.low:.1%} to {interval.high:.1%})")

**You should now see** `judge flags 17/150 sampled traces = 11.3%`, corrected to `0.084 [0.028, 0.144]`, against a true rate today of `8.0%`; then `alert vs baseline 10%: False`, because the judge-rate interval (7.2% to 17.4%) starts below the baseline.

### What just happened?

An alert on the point estimate ("11% today vs 10% last release!") fires on noise every other day, and people learn to ignore it. Alerting on the lower bound fires when the change is real at the sample size you're paying for. Compare like with like: the judge's raw rate today against the **same judge's** raw rate at the baseline (a consistent instrument), and report the corrected rate (the honest number). Comparing a raw judge rate with a true rate would have fired here: the judge over-flags.

> ⚠️ **Production note: sample online judges by design, not by default.**
> Pick the sample size from the margin you need, sample by trace id, cap judge spend (LangSmith has a weekly spend limit per evaluator), alert on interval bounds rather than point estimates, and re-validate the judge on freshly reviewed traces every month.

---
## Part 5 · The review queue, and back into the taxonomy

A review queue gets three streams: every 👎 (users noticed), every judge flag (the judge noticed), and a small **random** slice (nobody noticed: the only stream that finds new failure modes). A reviewer labels each item with a taxonomy category, or proposes a new one.

In [ ]:
# 🧪 Offline: fill a queue from three streams, label it, export examples for the next dataset version.
from mastery.evals import Dataset
from mastery.evals.monitoring import ReviewQueue

by_id = {t["id"]: t for t in production}
queue = ReviewQueue("desk-daily-review")
for f in feedback:
    if f["score"] == 0:
        queue.add(by_id[f["run_id"]], "thumbs_down")
for t, flagged in zip(sampled, flags):
    if flagged:
        queue.add(t, "judge:stopped_at_lookup")
for t in production:
    if should_sample(t["id"], 0.03, salt="random-review"):
        queue.add(t, "random")

print(len(queue.items), "items; by reason:", dict(Counter(i["reason"] for i in queue.items.values())))
for trace_id in list(queue.items):                       # the reviewer (here: the reference labeller)
    queue.label(trace_id, truth[trace_id], note="daily review")
print("labels:", queue.counts())

new_examples = queue.to_examples()
dev_only = Dataset("orbita-desk-production", [{**e, "split": "dev"} for e in new_examples])
print(f"\n{len(new_examples)} examples for the next dataset version (dev split only), version {dev_only.version}")

**You should now see** `67 items; by reason: {'thumbs_down': 44, 'judge:stopped_at_lookup': 15, 'random': 8}`, labels led by `asks_for_known_id` (27) and `stopped_at_lookup` (23), 7 items that were actually fine (a judge false alarm or an unfair 👎), and 67 examples exported under a new dataset version.

### What just happened?

- Thumbs-down and judge flags find **known** problems cheaply; the random slice is how you notice a category your taxonomy doesn't have yet (Lesson 10.01's saturation check, repeated on live traffic).
- Labelled items go into the **dev** split: the test split stays frozen for the current release cycle (Lesson 10.03), and the next dataset version gets a new hash.
- Labels on judge-flagged items double as fresh validation data for the judge (Part 4).

---
## Part 6 · The same loop in LangSmith

> 🌐 Talks to LangSmith (free Developer plan is enough; no model tokens). Without `LANGSMITH_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 🌐 LangSmith: trace one turn, attach user feedback to it, and put it in an annotation queue.
if HAVE_LANGSMITH_KEY:
    import uuid

    from langsmith import Client, traceable, tracing_context

    client = Client()
    run_id = uuid.uuid4()

    @traceable(name="desk_turn")
    def desk_turn(message):
        return {"answer": run_traces(app, [{"id": "live", "kind": "realistic", "want": ["refund"], "turns": [message], "decision": "approve"}], desk)[0]["answer"]}

    with tracing_context(enabled=True):
        desk_turn("I'm C-1003, please refund INV-3001, it was a duplicate charge.", langsmith_extra={"run_id": run_id})
    client.flush()
    client.create_feedback(**feedback_payload(str(run_id), "user_thumbs", False, comment="only listed invoices", source="user"))
    queue_ls = client.create_annotation_queue(name=f"m10-desk-review-{uuid.uuid4().hex[:6]}",
                                              description="Thumbs-down and judge-flagged desk traces")
    client.add_runs_to_annotation_queue(queue_ls.id, run_ids=[run_id])
    print("feedback attached; queue:", queue_ls.name)
else:
    print("⏭️ Skipped: LANGSMITH_API_KEY is not set (see the setup cell).")

**You should now see** `⏭️ Skipped` without a key. With one, the queue name; open **Annotation queues** in LangSmith to review the run, and the trace's **Feedback** tab to see the 👎.

The online judge itself is configured in the UI: **Tracing project → Evaluators → + Evaluator**, with a filter (for example, only traces that called `lookup_invoices`), a **sampling rate** and, under Advanced, a spend limit and the retention setting. Dashboards chart feedback scores over time; alerts notify you when a metric crosses a threshold.

> ⚠️ **Production note: online evaluators extend retention by default.**
> An online evaluator or automation rule upgrades every trace it touches to extended retention (and, for thread-level rules, every trace in the thread) unless you opt out on that evaluator; SDK feedback does so only with `extend_trace_retention=True`; UI feedback and manual queue adds don't (checked 2026-10-15). Decide per evaluator, and keep the traces you need forever in datasets, which never expire.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| "Failure rate" from thumbs is 3× what reviewers find | Feedback is self-selected | Use 👎 as pointers; rates from a random judged sample |
| Alert fires daily, nobody reacts | Alerting on point estimates | Alert on the interval bound (`rate_alert`) |
| Trace bill jumped after adding an evaluator | Retention auto-upgrade on judged traces | Opt out per evaluator, lower the sampling rate |
| Two workers judge different traces for the same rate | Random sampling per process | Hash-based `should_sample(trace_id, rate)` |
| Queue only shows known failures | No random slice | Add a small random stream |
| Test scores drift upward month after month | Production examples added to the test split | New examples go to dev; freeze test per release |

## ⚠️ Production notes

> ⚠️ **Production note: sample online judges by design, not by default.** (Part 4)

> ⚠️ **Production note: online evaluators extend retention by default.** (Part 6)

## 🏋️ Exercises

1. **(Easy)** What sample size gives ±3% on a failure rate near 10%? And near 50%?
2. **(Medium)** Change the feedback simulation so unhappy customers click 50% of the time. How does the thumbs-down share change? Does the true failure rate?
3. **(Medium)** Add a fourth queue stream: traces where the desk visited two or more specialists (`t["route"]` lists them). How many **new** items does it add?
4. **(Stretch)** Compute the judge's TPR/TNR on the **queue's** judge-flagged items only. Why can't you estimate TPR from them?

<details>
<summary><b>Show solutions</b></summary>

**1.** `sample_size(0.03, p=0.1)` → 385; `sample_size(0.03, p=0.5)` → 1068. Rates near 50% need the most data.

**2.** Change `0.30` to `0.50`: more feedback and an even higher thumbs-down share; the true rate is unchanged (`truth` doesn't depend on clicks). Feedback measures who clicks, not how often the desk fails.

**3.**
```python
before = len(queue.items)
for t in production:
    if len(t["route"]) >= 2:
        queue.add(t, "multi_specialist")
print(len(queue.items) - before, "new items;", Counter(i["reason"] for i in queue.items.values()))
```
`add` ignores traces that are already queued, so only traces no other stream picked are new. Multi-specialist turns are where `dropped_request` lives.

**4.** Every judge-flagged item is a predicted failure: you see true and false positives, but never the failures the judge **missed** (false negatives), so TPR = TP / (TP + FN) is not computable. Precision is. TPR needs a sample that doesn't depend on the judge's verdict, like the random stream.
</details>

## 🔎 Docs drill (5 min)

Which LangSmith actions upgrade a trace to extended retention, and which don't?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Usage and billing](https://docs.langchain.com/langsmith/usage-and-billing) → *Data retention auto-upgrades*.
2. Upgrade (by default, opt-out available): online evaluators that score the trace, automation rules that match a run in it (thread rules upgrade the whole thread), and SDK/API feedback sent with `extend_trace_retention=true`.
3. No upgrade: UI feedback and notes, and manually adding runs to an annotation queue. Experiments are created at extended retention from the start.

</details>

## ✅ Checkpoint

- [ ] I know what user feedback, online judges and reviewers each measure, and their biases.
- [ ] I sized a judge sample from the margin I need and estimated its daily cost.
- [ ] My alert compares an interval bound with the baseline.
- [ ] My review queue has a random stream, and labelled items flow into the dev split and the taxonomy.
- [ ] I know which LangSmith actions extend trace retention.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check [Online evaluators](https://docs.langchain.com/langsmith/online-evaluations) and [Usage and billing](https://docs.langchain.com/langsmith/usage-and-billing) and log in [`docs/research-log.md`](../../docs/research-log.md): the retention-upgrade rules (2026-10-15: on by default for online evaluators and rules, opt-out per evaluator) and whether decision-model judges are available on your plan.

## Summary

- Offline evals decide releases; online evals watch production and feed the next error analysis.
- User feedback is a pointer, not a rate; rates come from random samples, judged and corrected.
- Size samples from the margin you need; alert on interval bounds.
- Review queues need a random stream; labels flow into the taxonomy and the dev split.
- Online evaluators cost judge calls and extended retention; configure both on purpose.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 10-07-prompt-hub.ipynb** → [`10-07-prompt-hub.ipynb`](10-07-prompt-hub.ipynb).

[Module 10 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)